In [1]:
from google.colab import files

uploaded = files.upload()

Saving HR_Attrition_Analysis_Final.xlsx to HR_Attrition_Analysis_Final.xlsx


In [2]:
import pandas as pd

df = pd.read_excel(list(uploaded.keys())[0])

df.head()


,Age,Attrition,BusinessTravel,DailyRate,Department,DistanceFromHome,Education,EducationField,EmployeeCount,EmployeeNumber,...,StandardHours,StockOptionLevel,TotalWorkingYears,TrainingTimesLastYear,WorkLifeBalance,YearsAtCompany,YearsInCurrentRole,YearsSinceLastPromotion,YearsWithCurrManager,Age_Group
0,41,Yes,Travel_Rarely,1102,Sales,1,2,Life Sciences,1,1,...,80,0,8,0,1,6,4,0,5,41-50
1,49,No,Travel_Frequently,279,Research & Development,8,1,Life Sciences,1,2,...,80,1,10,3,3,10,7,1,7,41-50
2,37,Yes,Travel_Rarely,1373,Research & Development,2,2,Other,1,4,...,80,0,7,3,3,0,0,0,0,30-40
3,33,No,Travel_Frequently,1392,Research & Development,3,4,Life Sciences,1,5,...,80,0,8,3,3,8,7,3,0,30-40
4,27,No,Travel_Rarely,591,Research & Development,2,1,Medical,1,7,...,80,1,6,3,3,2,2,2,2,Under 30


In [4]:
import sqlite3

conn = sqlite3.connect(":memory:")

df.to_sql("hr", conn, index=False, if_exists="replace")


1470

In [5]:
query = """
SELECT
    Attrition,
    COUNT(*) AS Employee_Count
FROM hr
GROUP BY Attrition;
"""

pd.read_sql_query(query, conn)

,Attrition,Employee_Count
0,No,1233
1,Yes,237


In [6]:
query = """
SELECT
    Department,
    COUNT(*) AS Total_Employees,
    SUM(CASE WHEN Attrition = 'Yes' THEN 1 ELSE 0 END) AS Employees_Left,
    ROUND(
        SUM(CASE WHEN Attrition = 'Yes' THEN 1 ELSE 0 END) * 100.0 / COUNT(*),
        2
    ) AS Attrition_Rate
FROM hr
GROUP BY Department
ORDER BY Attrition_Rate DESC;
"""

pd.read_sql_query(query, conn)


,Department,Total_Employees,Employees_Left,Attrition_Rate
0,Sales,446,92,20.63
1,Human Resources,63,12,19.05
2,Research & Development,961,133,13.84


In [7]:
query = """
SELECT
    JobRole,
    COUNT(*) AS Total_Employees,
    SUM(CASE WHEN Attrition = 'Yes' THEN 1 ELSE 0 END) AS Employees_Left,
    ROUND(
        SUM(CASE WHEN Attrition = 'Yes' THEN 1 ELSE 0 END) * 100.0 / COUNT(*),
        2
    ) AS Attrition_Rate
FROM hr
GROUP BY JobRole
ORDER BY Attrition_Rate DESC;
"""

pd.read_sql_query(query, conn)

,JobRole,Total_Employees,Employees_Left,Attrition_Rate
0,Sales Representative,83,33,39.76
1,Laboratory Technician,259,62,23.94
2,Human Resources,52,12,23.08
3,Sales Executive,326,57,17.48
4,Research Scientist,292,47,16.10
5,Manufacturing Director,145,10,6.90
6,Healthcare Representative,131,9,6.87
7,Manager,102,5,4.90
8,Research Director,80,2,2.50


In [8]:
query = """
SELECT
    OverTime,
    COUNT(*) AS Total_Employees,
    SUM(CASE WHEN Attrition = 'Yes' THEN 1 ELSE 0 END) AS Employees_Left,
    ROUND(
        SUM(CASE WHEN Attrition = 'Yes' THEN 1 ELSE 0 END) * 100.0 / COUNT(*),
        2
    ) AS Attrition_Rate
FROM hr
GROUP BY OverTime
ORDER BY Attrition_Rate DESC;
"""

pd.read_sql_query(query, conn)


,OverTime,Total_Employees,Employees_Left,Attrition_Rate
0,Yes,416,127,30.53
1,No,1054,110,10.44


In [9]:
query = """
SELECT
    CASE
        WHEN Age < 30 THEN 'Under 30'
        WHEN Age BETWEEN 30 AND 40 THEN '30-40'
        WHEN Age BETWEEN 41 AND 50 THEN '41-50'
        ELSE '50+'
    END AS Age_Group,

    COUNT(*) AS Total_Employees,

    SUM(CASE WHEN Attrition = 'Yes' THEN 1 ELSE 0 END) AS Employees_Left,

    ROUND(
        SUM(CASE WHEN Attrition = 'Yes' THEN 1 ELSE 0 END) * 100.0 / COUNT(*),
        2
    ) AS Attrition_Rate

FROM hr

GROUP BY Age_Group

ORDER BY Attrition_Rate DESC;
"""

pd.read_sql_query(query, conn)

,Age_Group,Total_Employees,Employees_Left,Attrition_Rate
0,Under 30,326,91,27.91
1,30-40,679,94,13.84
2,50+,143,18,12.59
3,41-50,322,34,10.56


In [10]:
query = """
SELECT
    MaritalStatus,
    COUNT(*) AS Total_Employees,
    SUM(CASE WHEN Attrition = 'Yes' THEN 1 ELSE 0 END) AS Employees_Left,
    ROUND(
        SUM(CASE WHEN Attrition = 'Yes' THEN 1 ELSE 0 END) * 100.0 / COUNT(*),
        2
    ) AS Attrition_Rate
FROM hr
GROUP BY MaritalStatus
ORDER BY Attrition_Rate DESC;
"""

pd.read_sql_query(query, conn)

,MaritalStatus,Total_Employees,Employees_Left,Attrition_Rate
0,Single,470,120,25.53
1,Married,673,84,12.48
2,Divorced,327,33,10.09


In [12]:
query = """
WITH jobrole_attrition AS
(
    SELECT
        JobRole,
        COUNT(*) AS Total_Employees,
        SUM(CASE WHEN Attrition = 'Yes' THEN 1 ELSE 0 END) AS Employees_Left,
        ROUND(
            SUM(CASE WHEN Attrition = 'Yes' THEN 1 ELSE 0 END) * 100.0 / COUNT(*),
            2
        ) AS Attrition_Rate
    FROM hr
    GROUP BY JobRole
)

SELECT
    JobRole,
    Total_Employees,
    Employees_Left,
    Attrition_Rate,
    RANK() OVER(
        ORDER BY Attrition_Rate DESC
    ) AS Attrition_Rank
FROM jobrole_attrition;
"""

pd.read_sql_query(query, conn)

,JobRole,Total_Employees,Employees_Left,Attrition_Rate,Attrition_Rank
0,Sales Representative,83,33,39.76,1
1,Laboratory Technician,259,62,23.94,2
2,Human Resources,52,12,23.08,3
3,Sales Executive,326,57,17.48,4
4,Research Scientist,292,47,16.10,5
5,Manufacturing Director,145,10,6.90,6
6,Healthcare Representative,131,9,6.87,7
7,Manager,102,5,4.90,8
8,Research Director,80,2,2.50,9


In [11]:
query = """
SELECT
    Gender,
    COUNT(*) AS Total_Employees,
    SUM(CASE WHEN Attrition = 'Yes' THEN 1 ELSE 0 END) AS Employees_Left,
    ROUND(
        SUM(CASE WHEN Attrition = 'Yes' THEN 1 ELSE 0 END) * 100.0 / COUNT(*),
        2
    ) AS Attrition_Rate
FROM hr
GROUP BY Gender
ORDER BY Attrition_Rate DESC;
"""

pd.read_sql_query(query, conn)

,Gender,Total_Employees,Employees_Left,Attrition_Rate
0,Male,882,150,17.01
1,Female,588,87,14.80


In [13]:
query = """
WITH department_attrition AS
(
    SELECT
        Department,
        COUNT(*) AS Total_Employees,
        SUM(CASE WHEN Attrition = 'Yes' THEN 1 ELSE 0 END) AS Employees_Left,
        ROUND(
            SUM(CASE WHEN Attrition = 'Yes' THEN 1 ELSE 0 END) * 100.0 / COUNT(*),
            2
        ) AS Attrition_Rate
    FROM hr
    GROUP BY Department
)

SELECT
    Department,
    Total_Employees,
    Employees_Left,
    Attrition_Rate,
    RANK() OVER(
        ORDER BY Attrition_Rate DESC
    ) AS Department_Rank
FROM department_attrition;
"""

pd.read_sql_query(query, conn)

,Department,Total_Employees,Employees_Left,Attrition_Rate,Department_Rank
0,Sales,446,92,20.63,1
1,Human Resources,63,12,19.05,2
2,Research & Development,961,133,13.84,3


In [14]:
query = """
SELECT
    EmployeeNumber,
    JobRole,
    Department,
    MonthlyIncome,
    RANK() OVER(
        ORDER BY MonthlyIncome DESC
    ) AS Income_Rank
FROM hr
ORDER BY Income_Rank
LIMIT 10;
"""

pd.read_sql_query(query, conn)

,EmployeeNumber,JobRole,Department,MonthlyIncome,Income_Rank
0,259,Manager,Research & Development,19999,1
1,1035,Research Director,Research & Development,19973,2
2,1191,Manager,Research & Development,19943,3
3,226,Manager,Research & Development,19926,4
4,787,Manager,Research & Development,19859,5
5,1282,Manager,Sales,19847,6
6,1038,Manager,Sales,19845,7
7,1740,Manager,Sales,19833,8
8,1255,Research Director,Research & Development,19740,9
9,1338,Manager,Human Resources,19717,10


In [15]:
query = """
SELECT
    JobRole,
    COUNT(*) AS Total_Employees,
    ROUND(AVG(MonthlyIncome),2) AS Avg_Monthly_Income
FROM hr
GROUP BY JobRole
ORDER BY Avg_Monthly_Income DESC;
"""

pd.read_sql_query(query, conn)

,JobRole,Total_Employees,Avg_Monthly_Income
0,Manager,102,17181.68
1,Research Director,80,16033.55
2,Healthcare Representative,131,7528.76
3,Manufacturing Director,145,7295.14
4,Sales Executive,326,6924.28
5,Human Resources,52,4235.75
6,Research Scientist,292,3239.97
7,Laboratory Technician,259,3237.17
8,Sales Representative,83,2626.00


In [16]:
query = """
WITH jobrole_salary AS
(
    SELECT
        JobRole,
        COUNT(*) AS Total_Employees,
        ROUND(AVG(MonthlyIncome),2) AS Avg_Monthly_Income
    FROM hr
    GROUP BY JobRole
)

SELECT
    JobRole,
    Total_Employees,
    Avg_Monthly_Income,
    RANK() OVER(
        ORDER BY Avg_Monthly_Income DESC
    ) AS Salary_Rank
FROM jobrole_salary;
"""

pd.read_sql_query(query, conn)

,JobRole,Total_Employees,Avg_Monthly_Income,Salary_Rank
0,Manager,102,17181.68,1
1,Research Director,80,16033.55,2
2,Healthcare Representative,131,7528.76,3
3,Manufacturing Director,145,7295.14,4
4,Sales Executive,326,6924.28,5
5,Human Resources,52,4235.75,6
6,Research Scientist,292,3239.97,7
7,Laboratory Technician,259,3237.17,8
8,Sales Representative,83,2626.00,9


In [17]:
query = """
WITH jobrole_summary AS
(
    SELECT
        JobRole,
        COUNT(*) AS Total_Employees,
        SUM(CASE WHEN Attrition = 'Yes' THEN 1 ELSE 0 END) AS Employees_Left,
        ROUND(
            SUM(CASE WHEN Attrition = 'Yes' THEN 1 ELSE 0 END) * 100.0 / COUNT(*),
            2
        ) AS Attrition_Rate,
        ROUND(AVG(MonthlyIncome),2) AS Avg_Monthly_Income
    FROM hr
    GROUP BY JobRole
)

SELECT
    JobRole,
    Total_Employees,
    Employees_Left,
    Attrition_Rate,
    Avg_Monthly_Income,
    RANK() OVER(
        ORDER BY Attrition_Rate DESC
    ) AS Attrition_Rank,
    RANK() OVER(
        ORDER BY Avg_Monthly_Income DESC
    ) AS Salary_Rank
FROM jobrole_summary
ORDER BY Attrition_Rank;
"""

pd.read_sql_query(query, conn)

,JobRole,Total_Employees,Employees_Left,Attrition_Rate,Avg_Monthly_Income,Attrition_Rank,Salary_Rank
0,Sales Representative,83,33,39.76,2626.00,1,9
1,Laboratory Technician,259,62,23.94,3237.17,2,8
2,Human Resources,52,12,23.08,4235.75,3,6
3,Sales Executive,326,57,17.48,6924.28,4,5
4,Research Scientist,292,47,16.10,3239.97,5,7
5,Manufacturing Director,145,10,6.90,7295.14,6,4
6,Healthcare Representative,131,9,6.87,7528.76,7,3
7,Manager,102,5,4.90,17181.68,8,1
8,Research Director,80,2,2.50,16033.55,9,2
